In [ ]:
#%pip install -q -r requirements.txt

In [ ]:
import os
import random
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import classification_report,multilabel_confusion_matrix,f1_score,precision_score,recall_score

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

from transformers import AutoTokenizer,AutoModel,get_linear_schedule_with_warmup

import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
SEED = 42

DATA_PATH = "nepali_news_bias.csv"

MODEL_NAME = "Rajan/NepaliBERT"

OUTPUT_DIR = "./nepali_bert_bias_model"

MAX_LENGTH = 128

BATCH_SIZE = 16

EPOCHS = 10

LEARNING_RATE = 2e-5

WEIGHT_DECAY = 0.01

PATIENCE = 2

NUM_LABELS = 4

TARGETS = ["Political","Sensationalism","Framing","Stereotyping"]

TEXT_COL = "Headline"



In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
df = df[[TEXT_COL,*TARGETS]].copy()
df

In [ ]:
df = df[[TEXT_COL,*TARGETS]].copy()
df

In [ ]:
for target in TARGETS:
    print(df[target].value_counts())

In [ ]:
X = df[TEXT_COL].values
y = df[TARGETS].values

def multilabel_split(X, y, test_size, random_state):
    splitter = MultilabelStratifiedShuffleSplit(n_splits=1,test_size=test_size,random_state=random_state)
    return next(splitter.split(X, y))


train_val_idx, test_idx = multilabel_split(X, y,test_size=0.15,random_state=SEED)
train_val_df = df.iloc[train_val_idx]
test_df = df.iloc[test_idx]

X_train_val = train_val_df[TEXT_COL].values
y_train_val = train_val_df[TARGETS].values

train_idx, val_idx = multilabel_split(X_train_val, y_train_val,test_size=15 / 85,random_state=SEED)

train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)
test_df = test_df.reset_index(drop=True)


print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


In [ ]:
class NewsBiasDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length, targets):

        self.texts = dataframe[TEXT_COL].tolist()
        self.labels = torch.tensor(dataframe[targets].values,dtype=torch.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = self.tokenizer(
            self.texts[idx],
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt" )

        return {"input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": self.labels[idx]}


In [ ]:
train_dataset = NewsBiasDataset(train_df, tokenizer, MAX_LENGTH, TARGETS)
val_dataset   = NewsBiasDataset(val_df, tokenizer, MAX_LENGTH, TARGETS)
test_dataset  = NewsBiasDataset(test_df, tokenizer, MAX_LENGTH, TARGETS)


In [ ]:
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)



In [ ]:
class NepaliBERTMultiLabel(nn.Module):

    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name,add_pooling_layer=False)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.bert.config.hidden_size,num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids,attention_mask=attention_mask)

        cls_embedding = outputs.last_hidden_state[:, 0, :]

        logits = self.classifier(self.dropout(cls_embedding))

        return logits


In [ ]:
model = NepaliBERTMultiLabel(MODEL_NAME,NUM_LABELS)

model.to(device)
print("Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)


In [ ]:
train_labels = train_df[TARGETS].values
pos_counts = train_labels.sum(axis=0)
neg_counts = (len(train_labels) - pos_counts)
pos_weights = (neg_counts / np.maximum(pos_counts,1 ))

print("\nPositive class weights:")
for target, weight in zip(TARGETS,pos_weights):
    print(target,":",round(float(weight), 3))

pos_weights = torch.tensor(pos_weights,dtype=torch.float).to(device)

In [ ]:
criterion = nn.BCEWithLogitsLoss( pos_weight=pos_weights)



In [ ]:
optimizer = torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)


In [ ]:
total_training_steps = (len(train_loader) * EPOCHS)
scheduler = get_linear_schedule_with_warmup(optimizer,num_warmup_steps=int(0.1 * total_training_steps),
    num_training_steps=total_training_steps)


In [ ]:
def evaluate(model,loader,threshold=0.5):

    model.eval()

    all_logits = []

    all_labels = []

    total_loss = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            logits = model(input_ids,attention_mask)
            loss = criterion(logits,labels)
            total_loss += loss.item()
            all_logits.append(logits.cpu())
            all_labels.append(labels.cpu())
    all_logits = torch.cat(all_logits).numpy()
    all_labels = torch.cat(all_labels).numpy()
    probabilities = ( 1 / ( 1 + np.exp(-all_logits)))
    predictions = (probabilities >= threshold).astype(int)
    macro_f1 = f1_score(all_labels,predictions,average="macro",zero_division=0)


    return {
        "loss":
            total_loss / len(loader),

        "macro_f1":
            macro_f1,

        "labels":
            all_labels,

        "probabilities":
            probabilities,

        "predictions":
            predictions
    }


In [ ]:
best_f1 = 0
best_model_state = None
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        logits = model(input_ids,attention_mask)
        loss = criterion(logits,labels)
        loss.backward()

        # Gradient clipping

        torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_train_loss += loss.item()


    train_loss = (total_train_loss / len(train_loader))


    validation = evaluate( model,val_loader,threshold=0.5)
    val_loss = validation["loss"]
    val_f1 = validation["macro_f1"]

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    print(f"Train Loss: {train_loss:.4f}")

    print(f"Val Loss:   {val_loss:.4f}")

    print(f"Val Macro-F1: {val_f1:.4f}")


    if val_f1 > best_f1:
        best_f1 = val_f1
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
        print("✓ Best model updated")

    else:
        patience_counter += 1
        print(f"No improvement "f"({patience_counter}/{PATIENCE})")

    if patience_counter >= PATIENCE:
        print("\nEarly stopping.")

        break



In [ ]:
model.load_state_dict(best_model_state)
print("\nBest validation Macro-F1:",best_f1)

# Instead of assuming 0.5 is optimal for every bias type,
# find the best threshold on the validation set.
validation = evaluate(model,val_loader,threshold=0.5)

val_labels = validation["labels"]
val_probabilities = validation["probabilities"]
thresholds = np.arange(0.10,0.91,0.05)


best_thresholds = []
for i, target in enumerate(TARGETS):
    best_target_f1 = 0
    best_threshold = 0.5
    for threshold in thresholds:
        predictions = (val_probabilities[:, i]>= threshold).astype(int)
        score = f1_score(val_labels[:, i],predictions,zero_division=0)

        if score > best_target_f1:

            best_target_f1 = score

            best_threshold = threshold

    best_thresholds.append(best_threshold)

    print(f"{target}: "f"threshold={best_threshold:.2f}, "f"validation F1={best_target_f1:.4f}")


best_thresholds = np.array(best_thresholds)


In [ ]:
test_results = evaluate( model,test_loader,threshold=0.5)
test_labels = test_results["labels"]
test_probabilities = (test_results["probabilities"])
test_predictions = np.zeros_like(test_probabilities,dtype=int)
for i in range(NUM_LABELS):
    test_predictions[:, i] = (test_probabilities[:, i]>= best_thresholds[i]).astype(int)


In [ ]:
for i, target in enumerate(TARGETS):
    print(target)
    print(classification_report(test_labels[:, i],test_predictions[:, i],target_names=["No Bias","Bias"],digits=4,zero_division=0))


In [ ]:
micro_f1 = f1_score(
    test_labels,
    test_predictions,
    average="micro",
    zero_division=0
)


macro_f1 = f1_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0
)


weighted_f1 = f1_score(
    test_labels,
    test_predictions,
    average="weighted",
    zero_division=0
)


micro_precision = precision_score(
    test_labels,
    test_predictions,
    average="micro",
    zero_division=0
)


micro_recall = recall_score(
    test_labels,
    test_predictions,
    average="micro",
    zero_division=0
)

print(f"Micro Precision: {micro_precision:.4f}")
print(f"Micro Recall:    {micro_recall:.4f}")
print(f"Micro F1:        {micro_f1:.4f}")
print(f"Macro F1:        {macro_f1:.4f}")
print(f"Weighted F1:     {weighted_f1:.4f}")



In [ ]:
results = []
for i, target in enumerate(TARGETS):

    precision = precision_score(
        test_labels[:, i],
        test_predictions[:, i],
        zero_division=0
    )


    recall = recall_score(
        test_labels[:, i],
        test_predictions[:, i],
        zero_division=0
    )


    f1 = f1_score(
        test_labels[:, i],
        test_predictions[:, i],
        zero_division=0
    )


    results.append({
        "Target": target,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Threshold": best_thresholds[i]
    })


results_df = pd.DataFrame(results)
display(results_df.round(4))


In [ ]:
cm = multilabel_confusion_matrix(test_labels,test_predictions)
fig, axes = plt.subplots(2,2,figsize=(10, 8))
for i, target in enumerate(TARGETS):
    ax = axes.flat[i]
    sns.heatmap(cm[i],annot=True,fmt="d",cmap="Blues",cbar=False,ax=ax)
    ax.set_title(target)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticklabels(["No Bias", "Bias"])
    ax.set_yticklabels(["No Bias", "Bias"])

plt.tight_layout()
plt.show()



In [ ]:
predictions_df = test_df[[TEXT_COL]].copy()
for i, target in enumerate(TARGETS):
    predictions_df[f"{target}_Actual"] = test_labels[:, i]
    predictions_df[f"{target}_Probability"] = test_probabilities[:, i]
    predictions_df[f"{target}_Predicted"] = test_predictions[:, i]

predictions_df.to_csv("test_predictions.csv",index=False)
print("\nSaved: test_predictions.csv")


In [ ]:
results_df.to_csv("bias_detection_results.csv",index=False)


In [ ]:
#SAVE MODEL
os.makedirs(OUTPUT_DIR,exist_ok=True)
model.bert.save_pretrained(OUTPUT_DIR)

tokenizer.save_pretrained(OUTPUT_DIR)


torch.save({
        "classifier_state_dict":
            model.classifier.state_dict(),

        "thresholds":
            best_thresholds,

        "targets":
            TARGETS,

        "max_length":
            MAX_LENGTH
    },
    os.path.join(OUTPUT_DIR,"bias_classifier.pt")
)
print("\nModel saved to:",OUTPUT_DIR)




In [ ]:
def predict_bias(headline):

    model.eval()

    encoding = tokenizer(
        headline,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_tensors="pt")


    input_ids = encoding["input_ids"].to(device)


    attention_mask = encoding["attention_mask"].to(device)


    with torch.no_grad():
        logits = model(input_ids,attention_mask)

    probabilities = torch.sigmoid(logits).cpu().numpy()[0]

    predictions = (probabilities>= best_thresholds).astype(int)


    result = {}


    for i, target in enumerate(TARGETS):

        result[target] = {
            "probability":
                float(probabilities[i]),

            "prediction":
                int(predictions[i]),

            "threshold":
                float(best_thresholds[i])}
    return result




In [ ]:
example_headline = ("सरकारको नयाँ निर्णयप्रति विपक्षी दलको कडा विरोध")
prediction = predict_bias(example_headline)
print(example_headline)
print("\nPrediction:",prediction)

for target, result in prediction.items():
    print(f"{target}: "f"probability={result['probability']:.4f}, "f"prediction={result['prediction']}")